In [1]:
import sys
print(sys.executable)

c:\Users\admin\anaconda3\envs\rag_env\python.exe


In [1]:

import langchain
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings


from dotenv import load_dotenv
import os

print("All working 🚀")

c:\Users\admin\anaconda3\envs\rag_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All working 🚀


In [2]:
load_dotenv()

True

In [3]:
## Lets read document
def read_document(file_path):
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    return documents

In [4]:
docs = read_document("document/dsa.pdf")

print(len(docs))  

112


In [5]:
def chunk_data(docs, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = text_splitter.split_documents(docs)
    return chunks

In [6]:
chunks = chunk_data(docs)
chunks



[Document(metadata={'producer': 'dvipdfm 0.13.2d, Copyright © 1998, by Mark A. Wicks', 'creator': 'TeX output 2008.12.18:2245', 'creationdate': '2008-12-18T22:46:33+10:00', 'source': 'document/dsa.pdf', 'total_pages': 112, 'page': 0, 'page_label': '1'}, page_content='DSA\nDat a St ruc tur es and Alg orith ms\nAnn otated Referenc e w ith  Examp les\nGranville Bar ne/g425 Luca  Del Tongo'),
 Document(metadata={'producer': 'dvipdfm 0.13.2d, Copyright © 1998, by Mark A. Wicks', 'creator': 'TeX output 2008.12.18:2245', 'creationdate': '2008-12-18T22:46:33+10:00', 'source': 'document/dsa.pdf', 'total_pages': 112, 'page': 1, 'page_label': '2'}, page_content='Data Structures and Algorithms:\nAnnotated Reference with Examples\nFirst Edition\nCopyright c⃝ Granville Barnett, and Luca Del Tongo 2008.'),
 Document(metadata={'producer': 'dvipdfm 0.13.2d, Copyright © 1998, by Mark A. Wicks', 'creator': 'TeX output 2008.12.18:2245', 'creationdate': '2008-12-18T22:46:33+10:00', 'source': 'document/dsa.

In [7]:
print(len(chunks))

231


In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectors = embeddings.embed_query("how are you")

print(len(vectors))

C:\Users\admin\AppData\Local\Temp\ipykernel_13876\3550112751.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3578.08it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


384


In [10]:
vectors = embeddings.embed_query("hello")
vectors

[-0.06277178227901459,
 0.054958805441856384,
 0.05216481164097786,
 0.08579001575708389,
 -0.08274892717599869,
 -0.07457296550273895,
 0.06855472177267075,
 0.018396398052573204,
 -0.08201129734516144,
 -0.037384841591119766,
 0.012124874629080296,
 0.0035182819701731205,
 -0.004134260583668947,
 -0.04378441348671913,
 0.02180727943778038,
 -0.005102706141769886,
 0.01954660750925541,
 -0.04234874248504639,
 -0.11035963147878647,
 0.005424555856734514,
 -0.05573475733399391,
 0.02805243991315365,
 -0.023158736526966095,
 0.028481358662247658,
 -0.05370962247252464,
 -0.052601587027311325,
 0.03393925353884697,
 0.04538864642381668,
 0.023718424141407013,
 -0.07312081009149551,
 0.05477769672870636,
 0.017047282308340073,
 0.08136037737131119,
 -0.002862716792151332,
 0.011958083137869835,
 0.07355854660272598,
 -0.09423746168613434,
 -0.0813620537519455,
 0.04001539200544357,
 0.0006921681924723089,
 -0.013393286615610123,
 -0.05453810468316078,
 0.00515141012147069,
 -0.026139806956

In [11]:
len(vectors)

384

In [12]:
from pinecone import Pinecone


In [13]:
pc = Pinecone(api_key="pcsk_6qRSUr_Mx3M9hswYmZxF9cQaApWH72iCN3xzwqijEWFvepcWVLH4nQx6TuqMy3BWbxtiQX")
index = pc.Index("mentor")

In [14]:
vectors = []

for i, chunk in enumerate(chunks):
    vector = embeddings.embed_query(chunk.page_content)
    
    vectors.append({
        "id": str(i),
        "values": vector,
        "metadata": {"text": chunk.page_content}
    })

index.upsert(vectors)

UpsertResponse(upserted_count=231, _response_info={'raw_headers': {'date': 'Thu, 02 Apr 2026 13:20:42 GMT', 'content-type': 'application/json', 'content-length': '21', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '530656', 'x-pinecone-request-latency-ms': '3208', 'x-envoy-upstream-service-time': '316', 'x-pinecone-response-duration-ms': '3210', 'grpc-status': '0', 'server': 'envoy'}})

In [21]:
def ask_question(query):
    # 1. convert query → vector
    query_vector = embeddings.embed_query(query)

    # 2. search Pinecone
    results = index.query(
        vector=query_vector,
        top_k=3,
        include_metadata=True
    )

    # 3. collect context
    context = " ".join([
        match["metadata"]["text"] for match in results["matches"]
    ])

    # 4. simple answer (no LLM yet)
    return context

In [ ]:
question = "What is binary search?"

answer = ask_question(question)

print(answer)

CHAPTER 10. SEARCHING 77
Figure 10.1: a) Search(12), b) Search(101)
1) algorithm ProbabilitySearch(list, item)
2) Pre: list ̸= ∅
3) Post: a boolean indicating where the item is found or not;
in the former case swap founded item with its predecessor
4) index ← 0
5) while index < list .Count and list[index] ̸= item
6) index ← index + 1
7) end while
8) if index ≥ list.Count or list[index] ̸= item
9) return false
10) end if
11) if index > 0
12) Swap(list[index], list[index − 1])
13) end if
14) return true
15) end ProbabilitySearch
10.3 Summary
In this chapter we have presented a few novel searching algorithms. We have
presented more eﬃcient searching algorithms earlier on, like for instance the
logarithmic searching algorithm that A VL and BST tree’s use (deﬁned in §3.2).
We decided not to cover a searching algorithm known as binary chop (another
name for binary search, binary chop usually refers to its array counterpart) as Chapter 10
Searching
10.1 Sequential Search
A simple algorithm th